# 관람의향 설명 기여율과 소득집단 의향격차 분해

## 목적
- 네 장르에서 12개 설명변수의 4,096개 조합을 모두 적합
- 가중 로그우도 개선량과 소득집단 표준화 확률격차를 정확 Owen 방식으로 분해

같은 부분모형에서 두 값을 함께 계산해 중복 적합을 피한다.

In [ ]:
import pathlib
import sys
import pandas as pd

BASE_PATH = pathlib.Path.cwd().resolve()
if not (BASE_PATH / "data").exists():
    BASE_PATH = next(parent for parent in BASE_PATH.parents if (parent / "data").exists())
CODE_PATH = BASE_PATH / "code" / "01_eda"
if str(CODE_PATH) not in sys.path:
    sys.path.insert(0, str(CODE_PATH))
RAW_PATH = BASE_PATH / "data" / "raw"
PROCESSED_PATH = BASE_PATH / "data" / "processed" / "eda"
IMAGE_PATH = BASE_PATH / "output" / "image" / "readme"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.mkdir(parents=True, exist_ok=True)
print("BASE_PATH:", BASE_PATH)

## 입력 데이터와 실행 설정

In [ ]:
from analysis.models import analyze_models
INPUT_FILE = PROCESSED_PATH / "03_model_analysis_frame.parquet"
OUTPUT_DIR = PROCESSED_PATH / "02_models"
N_JOBS = 4
assert INPUT_FILE.exists()
model_frame = pd.read_parquet(INPUT_FILE)

## 4,096개 부분모형 적합과 Owen 분해

In [ ]:
outputs = analyze_models(model_frame, OUTPUT_DIR, jobs=N_JOBS)
intention = outputs["09_mean_intention_category_contribution.csv"]
gap = outputs["10_mean_gap_category_contribution.csv"]
display(intention.sort_values("네장르_동일가중평균_기여율_퍼센트", ascending=False).round(2))
display(gap.sort_values("네장르_동일가중평균_구성비_퍼센트", ascending=False).round(2))

## 결과 품질 점검

In [ ]:
subsets = outputs["02_all_subset_models.csv"]
assert len(subsets) == 4 * 4096 and subsets.groupby("장르").size().eq(4096).all()
assert abs(intention["네장르_동일가중평균_기여율_퍼센트"].sum() - 100) < 1e-7
assert abs(gap["네장르_동일가중평균_구성비_퍼센트"].sum() - 100) < 1e-7
display(outputs["07_gap_summary.csv"].round(3))

## 결과 요약
- 적합도 기여율과 소득격차 기여율은 서로 다른 질문에 답함
- 두 지표 모두 인과효과가 아님